In [39]:
!pip install -q -U langchain langchain-google-genai

#langchain -> agent + tools
#langchain-google-genai -> Gemini


In [57]:
import os
os.environ["GOOGLE_API_KEY"] = "YOUR API KEY HERE"

In [41]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

In [42]:
import sqlite3

conn = sqlite3.connect("students.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS students (
    student_id TEXT PRIMARY KEY,
    name TEXT,
    department TEXT,
    python INTEGER,
    database INTEGER,
    ai INTEGER,
    web INTEGER
)
""")

students = [
    ("22CS045", "Dhanushya", "Computer Science", 85, 72, 90, 78),
    ("22CS046", "Rahul", "Computer Science", 65, 70, 68, 72),
    ("22CS047", "Priya", "Information Technology", 92, 88, 95, 90),
    ("22CS048", "Arun", "Information Technology", 55, 60, 58, 62),
    ("22CS049", "Meena", "Computer Science", 78, 85, 80, 88)
]

cursor.executemany("""
INSERT OR REPLACE INTO students
(student_id, name, department, python, database, ai, web)
VALUES (?, ?, ?, ?, ?, ?, ?)
""", students)

conn.commit()
conn.close()

print("Database created successfully!")

Database created successfully!


In [43]:
#checking if db works or not

conn = sqlite3.connect("students.db")
cursor = conn.cursor()

cursor.execute("SELECT * FROM students")

rows = cursor.fetchall()

for row in rows:
    print(row)

conn.close()

('22CS045', 'Dhanushya', 'Computer Science', 85, 72, 90, 78)
('22CS046', 'Rahul', 'Computer Science', 65, 70, 68, 72)
('22CS047', 'Priya', 'Information Technology', 92, 88, 95, 90)
('22CS048', 'Arun', 'Information Technology', 55, 60, 58, 62)
('22CS049', 'Meena', 'Computer Science', 78, 85, 80, 88)


In [44]:
from langchain.tools import tool

In [45]:
#Student info

from langchain.tools import tool

@tool
def get_student_info(student_id: str) -> str:
    """Get the student's name and department using the student ID."""

    conn = sqlite3.connect("students.db")
    cursor = conn.cursor()

    cursor.execute("""
    SELECT name, department
    FROM students
    WHERE student_id = ?
    """, (student_id,))

    row = cursor.fetchone()

    conn.close()

    if row is None:
        return f"No student found with ID {student_id}"

    name, department = row

    return f"Name: {name}, Department: {department}"

In [46]:
#Student Marks

@tool
def get_student_marks(student_id: str) -> str:
    """Get the Python, Database, AI, and Web marks of a student."""

    conn = sqlite3.connect("students.db")
    cursor = conn.cursor()

    cursor.execute("""
    SELECT python, database, ai, web
    FROM students
    WHERE student_id = ?
    """, (student_id,))

    row = cursor.fetchone()

    conn.close()

    if row is None:
        return f"No student found with ID {student_id}"

    python_mark, database_mark, ai_mark, web_mark = row

    return (
        f"Python: {python_mark}, "
        f"Database: {database_mark}, "
        f"AI: {ai_mark}, "
        f"Web: {web_mark}"
    )

In [47]:
#Calculations

@tool
def calculator(expression: str) -> str:
    """Calculate a mathematical expression such as 85+72+90+78 or (85+72+90+78)/4."""

    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)

    except Exception:
        return "Invalid mathematical expression."

In [48]:
#Pass rules

@tool
def get_passing_rules() -> str:
    """Get the university passing rules for students."""

    return (
        "University passing rules: "
        "Minimum overall average is 40%. "
        "Minimum mark in each subject is 35%."
    )

In [49]:
@tool
def count_students_by_department(department: str) -> str:
    """Count how many students belong to a given department.
       Check carefully, CSE and Computer Science are same. The same way, IT and Information Technology are same.
       Think like that.
    """

    conn = sqlite3.connect("students.db")
    cursor = conn.cursor()

    cursor.execute("""
    SELECT COUNT(*)
    FROM students
    WHERE department = ?
    """, (department,))

    count = cursor.fetchone()[0]

    conn.close()

    return f"There are {count} students in the {department} department."

In [50]:
tools = [
    get_student_info,
    get_student_marks,
    calculator,
    get_passing_rules,
    count_students_by_department
]

for tool in tools:
    print(tool.name)

get_student_info
get_student_marks
calculator
get_passing_rules
count_students_by_department


In [51]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
You are a student information assistant.

Use the available tools whenever they are needed.

- Use get_student_info for student name and department.
- Use get_student_marks for student marks.
- Use calculator for total and average.
- Use get_passing_rules for passing requirements.
- Use count_students_by_department for student count.
- Do not guess information.
"""
)

In [54]:
def get_text(response):
    content = response["messages"][-1].content

    if isinstance(content, str):
        return content

    if isinstance(content, list):
        return "".join(
            item["text"]
            for item in content
            if isinstance(item, dict) and "text" in item
        )

    return str(content)

In [56]:
question = input("Ask your question: ")

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": question
        }
    ]
})

print("\nAgent:", get_text(response))

Ask your question: 22CS045

Agent: **Student Information:**
- **Name:** Dhanushya
- **Department:** Computer Science
- **Student ID:** 22CS045

**Marks:**
- **Python:** 85
- **Database:** 72
- **AI:** 90
- **Web:** 78

**Summary:**
- **Total Marks:** 325 / 400
- **Average Marks:** 81.25%
